<a href="https://colab.research.google.com/github/viktoriagajdosova/Enhancing-Meta-Research-in-Psychology-by-Generative-AI/blob/main/pipelines/02_topic-modeling/bertopic_meta-research_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bertopic openai umap-learn hdbscan pandas numpy
!pip install python-calamine

In [ ]:
# ==========================================================
# PART 1: BERTopic
# ==========================================================
import pandas as pd
import numpy as np
import os
import plotly.io as pio
from openai import OpenAI
from google.colab import userdata
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

# --- COLAB GRAPH FIX ---
from google.colab import output
output.enable_custom_widget_manager()
pio.renderers.default = "colab"

# --- 1. CONFIGURATION ---
excel_file = "dataset_IO.xlsx"
target_column = "06_abstract"
title_column = "03_title"
year_column = "07_year"
embedding_cache_file = "embeddings_openai_large.npy"

# --- 2. DATA LOADING ---
df = pd.read_excel(excel_file)
df = df[df[target_column].notna()].copy()
df = df.drop_duplicates(subset=[target_column]).copy()
docs = df[target_column].tolist()

# --- 3. EMBEDDINGS (OpenAI Version) ---
def get_openai_embeddings(text_list, model="text-embedding-3-large"):
    if os.path.exists(embedding_cache_file):
        print("Loading cached OpenAI embeddings...")
        return np.load(embedding_cache_file)

    print(f"Generating OpenAI embeddings for {len(text_list)} documents...")
    client = OpenAI(api_key=userdata.get('OPENAI_API_KEY2'))  # Make sure to set up your API key in Colab Secrets

    batch_size = 100.  # Process in batches to handle API limits
    all_embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = client.embeddings.create(input=batch, model=model)
        batch_embeddings = [data.embedding for data in response.data]
        all_embeddings.extend(batch_embeddings)

    embeddings = np.array(all_embeddings)
    np.save(embedding_cache_file, embeddings)
    return embeddings

embeddings = get_openai_embeddings(docs)

# --- 4. BERTopic SETUP (Auto-Reduction Mode + Advanced Cleaning) ---
# Note: we can change these parameters
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=40,
    min_samples=5,
    cluster_selection_method='eom',
    prediction_data=True
)

# --- CUSTOM TEXT CLEANING ---
academic_noise = list(CountVectorizer(stop_words="english").get_stop_words()) + \
                ["study", "research", "results", "participants", "analysis", "paper", "support", "model", "theory", "science", "journal",
                 "data", "findings", "abstract", "conclusions", "significant", "associated", "relationship", "effects", "psychology"]

vectorizer_model = CountVectorizer(stop_words=academic_noise, ngram_range=(1, 2))

# --- BETTER WORD WEIGHTING (c-TF-IDF) ---
ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics=None,
    verbose=True
)

topics, _ = topic_model.fit_transform(docs, embeddings)

# --- 5. VISUALIZATIONS ---
print(f"\nModel automatically converged to {len(topic_model.get_topics())} topics.")

def display_and_save(fig, name):
    fig.write_html(f"{name}.html")
    fig.show(renderer="colab")

# A. UMAP DOCUMENT ATLAS
print("Generating UMAP Document Atlas...")
fig_umap = topic_model.visualize_documents(docs, embeddings=embeddings, hide_annotations=True)
if title_column in df.columns:
    fig_umap.update_traces(hovertext=df[title_column].tolist())
display_and_save(fig_umap, "4_umap_document_atlas")

# B. Intertopic Distance Map
display_and_save(topic_model.visualize_topics(), "1_distance_map")

# C. Hierarchical Clustering
hierarchical_topics = topic_model.hierarchical_topics(docs)
display_and_save(topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics), "2_hierarchy")

# D. Topics Over Time
print("Displaying Trends...")
df[year_column] = pd.to_numeric(df[year_column], errors='coerce').fillna(0).astype(int)
topics_over_time = topic_model.topics_over_time(docs, df[year_column].tolist())
display_and_save(topic_model.visualize_topics_over_time(topics_over_time), "3_trends")

# E. TOPIC WORD SCORES (Barchart)
print("Generating Topic Word Scores Barchart...")
fig_barchart = topic_model.visualize_barchart(top_n_topics=200, n_words=7)
display_and_save(fig_barchart, "7_topic_word_scores")

# Save the model
topic_model.save("my_stable_model", serialization="safetensors", save_ctfidf=True)

In [ ]:
# ==========================================================
# PART 2: ADVANCED RESEARCH GAP & DYNAMICS ANALYSIS
# ==========================================================
import pandas as pd
import numpy as np
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import MaximalMarginalRelevance
from sklearn.metrics.pairwise import cosine_similarity

print("--- STARTING MULTI-DIMENSIONAL RESEARCH GAP ANALYSIS ---")

# --- 1. REFINING TOPIC REPRESENTATIONS ---
df['Topic_ID'] = topics
extra_stop_words = ["and", "of", "are", "the", "in", "to", "with", "study", "research",
                    "results", "is", "was", "for", "that", "this", "on", "as", "from",
                    "at", "by", "using", "method", "participants", "findings"]
stop_words = list(text.ENGLISH_STOP_WORDS.union(extra_stop_words))
representation_model = MaximalMarginalRelevance(diversity=0.5)

topic_model.update_topics(
    docs,
    vectorizer_model=CountVectorizer(stop_words=stop_words, ngram_range=(1, 2)),
    representation_model=representation_model
)

# --- [RESULT 1] BOTTOM-UP DISCOVERY: NOISE ANALYSIS ---
print("\n" + "="*60)
print("[RESULT 1] Analysis of Semantic Noise (Unclustered Papers)")
if -1 in topic_model.get_topics():
    noise_words = [word for word, score in topic_model.get_topic(-1)]
    print(f"Keywords hiding in the Noise (Topic -1): {noise_words[:15]}")
    gaps_df = df[df['Topic_ID'] == -1].copy()
    gaps_df.to_excel("result_1_bottom_up_gaps.xlsx", index=False)
    print(f"Outcome: {len(gaps_df)} papers identified as 'Semantic Noise' -> Saved to 'result_1_bottom_up_gaps.xlsx'.")

# --- [RESULT 2] THRESHOLD-BASED SEMANTIC ISOLATION ---
print("\n" + "="*60)
print("[RESULT 2] Threshold-Based Semantic Isolation")
iso_threshold = 0.99

if 'probs' not in locals() or probs is None:
    print("Notice: Probabilities not available. Filtering by Topic -1.")
    df_isolated = df[df['Topic_ID'] == -1].copy()
    df_isolated['Isolation_Score'] = 1.0
else:
    max_probs = np.max(probs, axis=1) if len(probs.shape) > 1 else probs
    df['Isolation_Score'] = 1 - max_probs
    df_isolated = df[df['Isolation_Score'] >= iso_threshold].copy()

total_isolated = len(df_isolated)
print(f"IDENTIFIED {total_isolated} PAPERS EXCEEDING {iso_threshold} THRESHOLD.")

if total_isolated > 0:
    t_col = next((c for c in df.columns if 'title' in c.lower()), 'Title')
    print(f"Previewing first 5 isolated papers:")
    for i, (idx, row) in enumerate(df_isolated.head(5).iterrows(), 1):
        print(f"  {i}. {row[t_col]} (Score: {round(row['Isolation_Score'], 4)})")
    df_isolated.to_excel("result_2_isolated_papers.xlsx", index=False)
    print(f"Outcome: Full list saved to 'result_2_isolated_papers.xlsx'.")
else:
    print(f"Outcome: No papers met the {iso_threshold} criteria.")

# --- [RESULT 3] RELATIONAL GAPS: INTER-TOPIC ISOLATION -----
print("\n" + "="*60)
print("[RESULT 3] Relational Gaps (Missing Connections)")

# Threshold for a "Weak Link" based on Cosine Similarity:
# $$\text{similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$
threshold = 0.65

topic_info = topic_model.get_topic_info()
valid_topics = topic_info[topic_info['Topic'] != -1].copy()
valid_indices = valid_topics.index.tolist()

# Get embeddings and calculate similarity matrix
valid_embeddings = [topic_model.topic_embeddings_[i] for i in valid_indices]
sim_matrix = cosine_similarity(valid_embeddings)

relational_gaps = []

# 1. Collect all gaps first without printing
for i in range(len(sim_matrix)):
    for j in range(i+1, len(sim_matrix)):
        if sim_matrix[i, j] < threshold:
            relational_gaps.append({
                "Topic_A_ID": valid_topics.iloc[i]['Topic'],
                "Topic_A_Name": valid_topics.iloc[i]['Name'],
                "Topic_B_ID": valid_topics.iloc[j]['Topic'],
                "Topic_B_Name": valid_topics.iloc[j]['Name'],
                "Similarity_Score": round(sim_matrix[i, j], 4)
            })

if relational_gaps:
    # 2. Sort by Similarity Score (ascending) - lowest similarity = biggest gap
    relational_gaps_sorted = sorted(relational_gaps, key=lambda x: x['Similarity_Score'])

    # 3. Print only the top 20 biggest gaps to console
    print(f"Top 20 Biggest Research Gaps (Lowest Similarity):")
    for gap in relational_gaps_sorted[:20]:
        print(f"- Massive Gap: [{gap['Topic_A_Name']}] <---> [{gap['Topic_B_Name']}] (Sim: {gap['Similarity_Score']})")

    # 4. Save ALL gaps to Excel
    pd.DataFrame(relational_gaps_sorted).to_excel("result_3_relational_gaps.xlsx", index=False)
    print(f"\nOutcome: {len(relational_gaps)} total gaps identified. Full list saved to 'result_3_relational_gaps.xlsx'.")
else:
    print(f"No gaps found at threshold {threshold}. Try increasing it.")

# --- [RESULT 4] TEMPORAL DYNAMICS: DECLINING vs. EMERGING ---
print("\n" + "="*60)
print("[RESULT 4] Temporal Dynamics (Pulse of the Field)")
median_year = df[year_column].median()
print(f"Analyzing shifts relative to median year: {int(median_year)}")

# Split the dataset into two chronological halves
era1_counts = df[df[year_column] <= median_year].groupby('Topic_ID').size()
era2_counts = df[df[year_column] > median_year].groupby('Topic_ID').size()

dynamics_report = []

for tid in topic_model.get_topic_info()['Topic']:
    if tid == -1: continue

    c1 = era1_counts.get(tid, 0)
    c2 = era2_counts.get(tid, 0)

    if c1 > 0 or c2 > 0:
        # Calculate percentage change
        change = ((c2 - c1) / c1 * 100) if c1 > 0 else 100.0
        topic_name = topic_model.get_topic_info(tid)['Name'].values[0]

        if change < -40:
            status = "📉 DECLINING (Temporal Gap)"
            print(f"Topic {tid} [{topic_name}]: {round(change, 1)}% -> {status}")
            dynamics_report.append({"Topic_ID": tid, "Name": topic_name, "Change_%": round(change, 2), "Status": "Declining"})

        elif change > 40:
            status = "🔥 EMERGING (Hot Topic)"
            print(f"Topic {tid} [{topic_name}]: +{round(change, 1)}% -> {status}")
            dynamics_report.append({"Topic_ID": tid, "Name": topic_name, "Change_%": round(change, 2), "Status": "Emerging"})

        else:
            status = "⚖️ STABLE"
            dynamics_report.append({"Topic_ID": tid, "Name": topic_name, "Change_%": round(change, 2), "Status": "Stable"})

# Export the complete movement report
if dynamics_report:
    pd.DataFrame(dynamics_report).to_excel("result_4_temporal_dynamics.xlsx", index=False)
    print(f"\nOutcome: Full dynamics report for all topics saved to 'result_4_temporal_dynamics.xlsx'.")

# --- [RESULT 6] FINAL MAPPING: EVERY ABSTRACT WITH ITS TOPIC ---
print("\n" + "="*60)
print("[RESULT 6] Creating Final Mapping Excel...")

# 1. Get the descriptive names for each topic
topic_info = topic_model.get_topic_info()[['Topic', 'Name']]

# 2. Ensure Topic_ID is in our main dataframe
df['Topic_ID'] = topics

# 3. Merge the descriptive names into our main dataframe
# This adds a 'Name' column so you see the keywords, not just a number
final_df = df.merge(topic_info, left_on='Topic_ID', right_on='Topic', how='left')

# 4. Clean up: Remove the redundant 'Topic' column from the merge
if 'Topic' in final_df.columns:
    final_df = final_df.drop(columns=['Topic'])

# 5. Save to Excel
final_output_name = "result_6_final_labeled_dataset.xlsx"
final_df.to_excel(final_output_name, index=False)

print(f"Outcome: Every abstract has been mapped to its topic.")
print(f"File saved as: '{final_output_name}'")

In [ ]:
# ==========================================================
# 🚀 APP: I-O PSYCHOLOGY RESEARCH GAP ANALYZER
# ==========================================================
import pandas as pd
import numpy as np
import os
from openai import OpenAI
from google.colab import userdata
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output
from bertopic import BERTopic

# --- 🛠️ CONFIGURATION ---
CONFIG = {
    "excel": "dataset_IO.xlsx",
    "embeddings": "embeddings_openai_large.npy",
    "model": "my_stable_model",
    "target_col": "06_abstract"
}

# --- 📋 SESSION HISTORY ---
history_log = []

# --- 🎨 USER INTERFACE ---
header = widgets.HTML("<h2>🔍 I-O Psychology Topic Probe</h2><p>Measuring alignment between new work era trends and established research themes.</p>")
text_input = widgets.Text(placeholder='e.g., Resilience, Care Economy...', description='Concept:', layout={'width': '500px'})
analyze_btn = widgets.Button(description='Analyze Topic Alignment', button_style='success', icon='search')
export_btn = widgets.Button(description='Export Results', button_style='info', icon='download')
output_area = widgets.Output()

def run_app_logic(btn):
    with output_area:
        clear_output()
        concept = text_input.value.strip()
        if not concept:
            print("⚠️ Please enter a theoretical construct.")
            return

        # 1. Loading Data & Model
        if not os.path.exists(CONFIG["model"]):
            print(f"❌ Error: BERTopic model not found at '{CONFIG['model']}'!")
            return

        df_internal = pd.read_excel(CONFIG["excel"])
        df_internal = df_internal[df_internal[CONFIG["target_col"]].notna()].copy()
        df_internal = df_internal.drop_duplicates(subset=[CONFIG["target_col"]]).copy()
        embeddings_internal = np.load(CONFIG["embeddings"])

        # Load the BERTopic model to access centroids
        tm = BERTopic.load(CONFIG["model"])

        # 2. Vectorization
        print(f"📡 Projecting '{concept}' into semantic space...")
        client = OpenAI(api_key=userdata.get('OPENAI_API_KEY2'))
        res = client.embeddings.create(input=[concept], model="text-embedding-3-large")
        concept_vec = np.array(res.data[0].embedding).reshape(1, -1)

        # 3. THEMATIC ALIGNMENT (Calculating distance to Closest Topic)
        # We skip the first embedding if it represents Topic -1 (Noise)
        topic_info = tm.get_topic_info()
        topic_embeddings = np.array(tm.topic_embeddings_)

        # Calculate similarity against all topic centroids
        topic_sims = cosine_similarity(concept_vec, topic_embeddings).flatten()

        # Find the best match (excluding noise Topic -1 if desired, usually index 0)
        best_topic_idx = np.argmax(topic_sims)
        overall_alignment_score = float(topic_sims[best_topic_idx])
        best_topic_name = topic_info.iloc[best_topic_idx]['Name']

        # 4. Threshold Interpretation
        if overall_alignment_score < 0.30:
            verdict, color = "CONFIRMED LATENT GAP", "🚨"
        elif 0.30 <= overall_alignment_score < 0.40:
            verdict, color = "MINIMAL REPRESENTATION", "🟡"
        else:
            verdict, color = "ESTABLISHED DISCOURSE", "✅"

# 5. Display Results
        print(f"\n{'='*70}\nCONSTRUCT ANALYSIS: '{concept.upper()}'\n{'='*70}")
        print(f"Closest Semantic Topic: {best_topic_name} (Sim: {round(overall_alignment_score, 4)})")
        print(f"Overall Alignment Status: {color} {verdict}")
        print(f"{'='*70}\n🔍 TOP 10 SUPPORTING SOURCES (Verification):")

        # Individual Document Similarity for context
        doc_sims = cosine_similarity(concept_vec, embeddings_internal).flatten()
        top_indices = doc_sims.argsort()[-10:][::-1]

        # Identify columns dynamically
        title_col = next((c for c in df_internal.columns if 'title' in c.lower()), 'Title')
        year_col = next((c for c in df_internal.columns if 'year' in c.lower()), 'Year')
        doi_col = next((c for c in df_internal.columns if 'doi' in c.lower()), 'DOI') # Added DOI search

        for i, idx in enumerate(top_indices, 1):
            row = df_internal.iloc[idx]
            # Added row.get(doi_col) to the print statement
            print(f"{i}. [{row.get(year_col)}] {row.get(title_col)}")
            print(f"   DOI: {row.get(doi_col)} | (Doc Sim: {round(doc_sims[idx], 3)})\n")

        # Log for export
        history_log.append({
            "Query": concept,
            "Closest_Topic": best_topic_name,
            "Alignment_Score": round(overall_alignment_score, 4),
            "Status": verdict
        })

analyze_btn.on_click(run_app_logic)
export_btn.on_click(lambda b: pd.DataFrame(history_log).to_excel("rq4_alignment_summary.xlsx", index=False))
display(header, widgets.VBox([text_input, widgets.HBox([analyze_btn, export_btn])]), output_area)